# Chart 1 — Subnet-rate rules timeline + geo view

VoltDB evaluates two independent **subnet-rate** rules per transaction. This chart shows both firing on a single `/24` subnet (`203.0.113.0/24`) over a 2-minute window:

- **`SUBNET_PAGE_HIT_RATE`** (`> 500 page hits / 5s window`) — tripped by a bot scanning public pages.
- **`SUBNET_TXN_RATE`** (`> 20 txn requests / 5s window`) — tripped by a credential-stuffing-style burst of transaction attempts from the same subnet.

Per-account rules (`VELOCITY_BURST`, `HIGH_SPEND`) are the focus of a separate chart: [Chart 2](chart2_txn_velocity_timeline.ipynb) shows how `SUBNET_TXN_RATE` and `VELOCITY_BURST` interact (each firing alone + both firing in the same window).

## Color scheme (consistent map ↔ timeline)

- 🟣 **Purple family** = page-hit side (light purple = normal, dark purple = exceeded).
- 🩵 **Cyan family** = txn-request side (light cyan = normal, dark cyan = exceeded).
- 🟢 **Green** = ACCEPTED transaction (solid fill).
- ⚪ **Grey with colored ring** = REJECTED transaction — ring color matches the signal family that caused the rejection (dark purple ring = page-hit reject, dark cyan ring = txn-count reject).
- 🩷 **Magenta** = highlight ring around the attacker subnet on the map.

## Top panel — world map

Each `/24` subnet with activity shows a **pair of markers side-by-side at its city** (scattergeo supports only circles — not real bars, so we approximate with two offset circles):
- **Purple marker** (west side), shade and size reflect **page-hit count**.
- **Cyan marker** (east side), shade and size reflect **transaction-request count**.

A transparent **magenta ring** + an **"⚠ ATTACKER" label** highlight the attacker subnet.

## Bottom panel — dual-axis timeline on the target subnet only

- **Left axis, LOG scale** — page hits per 5s window as a **light-purple step line**; dark-purple marker on windows that exceeded `SUBNET_PAGE_HIT_RATE`.
- **Right axis, LINEAR scale (0–30)** — txn requests per 5s window as a **light-cyan step line**; dark-cyan marker on exceeded windows.
- **Shaded regions** span contiguous bad windows: purple for page-rate exceeded, cyan for txn-rate exceeded.
- **Dashed threshold lines** — dark purple at 500 (left axis), dark cyan at 20 (right axis).
- **Dots** overlaid at each txn's time:
  - 🟢 ACCEPTED (solid green).
  - ⚪ 🟣 REJECTED by `SUBNET_PAGE_HIT_RATE` (grey, dark-purple ring).
  - ⚪ 🩵 REJECTED by `SUBNET_TXN_RATE` (grey, dark-cyan ring).

Hover any element for subnet / IP / amount / rule / reason details.

---

**This notebook uses INLINE FAKE DATA to iterate on the chart design.** Once the visual story is locked, we populate a BigQuery scratch table with equivalent data and swap the DataFrame construction for live BQ queries (see the commented-out `# BQ VARIANT` block in each query cell).

In [1]:
# !pip install pandas plotly  # google-cloud-bigquery db-dtypes needed later for BQ swap

In [2]:
import math
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, timezone

# Emit BOTH the plotly MIME output (interactive — Jupyter/IntelliJ with plotly
# plugin render natively) AND an HTML+plotly.js-via-CDN fallback (any static
# notebook viewer or exported HTML can display it).
pio.renderers.default = "plotly_mimetype+notebook_connected"

# ── Configuration ────────────────────────────────────────────────────────────
TARGET_SUBNET       = "78.46.220.0"       # the subnet under attack (/24)
PAGE_RULE_THRESHOLD = 500                 # SUBNET_PAGE_HIT_RATE: >N page hits per 5s → reject
TXN_RULE_THRESHOLD  = 20                  # SUBNET_TXN_RATE:      >N txn requests per 5s → reject

# Fixed anchor for reproducible fake data. Replace with real window bounds when
# the notebook pulls from BQ.
T0 = datetime(2026, 10, 3, 14, 0, 0, tzinfo=timezone.utc)   # window start

## Fake Query 1 — page-hit windows on `203.0.113.0`

Mirrors the BQ query that goes here:

```sql
SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(timestamp), 5) * 5) AS window_start,
       COUNT(*) AS hits
FROM `voltdb-operator.beam_threat_detection.page_hits`
WHERE subnet = '203.0.113.0'
  AND timestamp BETWEEN @start AND @end
GROUP BY window_start
ORDER BY window_start
```

The fake data below hand-crafts a believable timeline: ~30s of quiet baseline on this subnet (a few scattered hits per 5s window — one or two people browsing), then a 30-second bot burst that spikes bars well above the 500/5s `SUBNET_PAGE_HIT_RATE` threshold, then a tail-off.

In [3]:
# One row per 5-second window, 2 minutes of coverage (24 windows total).
# Windows 0..5   — baseline (3-8 hits per 5s)
# Windows 6..11  — PAGE BURST (bot scan: 120-1200 hits per 5s)
#                  windows 7-10 EXCEED PAGE_RULE_THRESHOLD (500) → bad windows
# Windows 12..23 — tail-off then baseline again (page hits settle under threshold)

hit_counts = [
    5, 3, 6, 4, 7, 5,                               # w0-5  baseline (0-30s)
    120, 850, 1200, 1100, 980, 450,                 # w6-11 burst (30-60s), w7-10 > 500
    80, 15, 8, 6, 4, 7, 5, 3, 6, 4, 8, 5,           # w12-23 tail + baseline (60-120s)
]
df_hits = pd.DataFrame({
    "window_start": [T0 + timedelta(seconds=5 * i) for i in range(len(hit_counts))],
    "hits":         hit_counts,
})
df_hits["page_exceeded"] = df_hits["hits"] > PAGE_RULE_THRESHOLD
print(f"{len(df_hits)} windows; {df_hits['page_exceeded'].sum()} windows exceed PAGE_RULE_THRESHOLD ({PAGE_RULE_THRESHOLD})")
df_hits.head(12)

# BQ VARIANT (uncomment + remove inline fake data when ready to swap):
#
# from google.cloud import bigquery
# client = bigquery.Client(project="voltdb-operator")
# df_hits = client.query(f"""
#     SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(timestamp), 5) * 5) AS window_start,
#            COUNT(*) AS hits
#     FROM `voltdb-operator.beam_threat_detection.page_hits`
#     WHERE subnet = '{TARGET_SUBNET}'
#       AND timestamp BETWEEN TIMESTAMP('{T0.isoformat()}')
#                         AND TIMESTAMP('{(T0 + timedelta(minutes=2)).isoformat()}')
#     GROUP BY window_start ORDER BY window_start
# """).to_dataframe()
# df_hits["page_exceeded"] = df_hits["hits"] > PAGE_RULE_THRESHOLD

24 windows; 4 windows exceed PAGE_RULE_THRESHOLD (500)


,window_start,hits,page_exceeded
0,2026-10-03 14:00:00+00:00,5,False
1,2026-10-03 14:00:05+00:00,3,False
2,2026-10-03 14:00:10+00:00,6,False
3,2026-10-03 14:00:15+00:00,4,False
4,2026-10-03 14:00:20+00:00,7,False
5,2026-10-03 14:00:25+00:00,5,False
6,2026-10-03 14:00:30+00:00,120,False
7,2026-10-03 14:00:35+00:00,850,True
8,2026-10-03 14:00:40+00:00,1200,True
9,2026-10-03 14:00:45+00:00,1100,True


## Fake Query 2 — transaction-request count per 5s window on `203.0.113.0/24`

Mirrors the BQ query that goes here (same 5s bucketing as page hits, filtered to the target subnet):

```sql
SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(txn_time), 5) * 5) AS window_start,
       COUNT(*) AS txn_count
FROM `voltdb-operator.beam_threat_detection.transactions`
WHERE source_ip LIKE '203.0.113.%'
  AND txn_time BETWEEN @start AND @end
GROUP BY window_start
ORDER BY window_start
```

Normal subnets see 0-5 transaction requests per 5-second window. The fake data below has baseline levels across most windows plus one window (`w15`, 75-80s) with a sudden spike to 24 requests — a credential-stuffing-style attack where many txn attempts arrive in quick succession from the same subnet. This window's count exceeds `SUBNET_TXN_RATE` (20/5s), triggering transaction rejections even though page-hit activity is quiet in that window.

In [4]:
# One row per 5-second window — transaction-request count on 78.46.220.0/24.
# Window 15 (75-80s) has a credential-stuffing-style burst of 24 txn requests,
# well above TXN_RULE_THRESHOLD (20). All other windows are at baseline (0-4).

txn_counts = [
    1, 2, 0, 1, 0, 2,          # w0-5  baseline
    1, 2, 0, 1, 2, 0,          # w6-11 during/after page burst (low txn — attack is scraping, not transacting)
    1, 1, 0,                   # w12-14 tail
    24,                        # w15 TXN BURST — above TXN_RULE_THRESHOLD
    0, 2, 1,                   # w16-18 recovery (w17 has one per-account VELOCITY_BURST)
    1, 0, 2, 1, 1,             # w19-23 baseline
]
assert len(txn_counts) == len(df_hits), "txn_counts must align with df_hits windows"
df_txn_windows = pd.DataFrame({
    "window_start": df_hits["window_start"],
    "txn_count":    txn_counts,
})
df_txn_windows["txn_exceeded"] = df_txn_windows["txn_count"] > TXN_RULE_THRESHOLD
print(f"{len(df_txn_windows)} windows; {df_txn_windows['txn_exceeded'].sum()} windows exceed TXN_RULE_THRESHOLD ({TXN_RULE_THRESHOLD})")
df_txn_windows[df_txn_windows["txn_exceeded"] | (df_txn_windows["txn_count"] > 0)].head(10)

# BQ VARIANT:
#
# df_txn_windows = client.query(f"""
#     SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(txn_time), 5) * 5) AS window_start,
#            COUNT(*) AS txn_count
#     FROM `voltdb-operator.beam_threat_detection.transactions`
#     WHERE source_ip LIKE '{".".join(TARGET_SUBNET.split(".")[:3])}.%'
#       AND txn_time BETWEEN TIMESTAMP('{T0.isoformat()}')
#                        AND TIMESTAMP('{(T0 + timedelta(minutes=2)).isoformat()}')
#     GROUP BY window_start ORDER BY window_start
# """).to_dataframe()
# df_txn_windows["txn_exceeded"] = df_txn_windows["txn_count"] > TXN_RULE_THRESHOLD

24 windows; 1 windows exceed TXN_RULE_THRESHOLD (20)


,window_start,txn_count,txn_exceeded
0,2026-10-03 14:00:00+00:00,1,False
1,2026-10-03 14:00:05+00:00,2,False
3,2026-10-03 14:00:15+00:00,1,False
5,2026-10-03 14:00:25+00:00,2,False
6,2026-10-03 14:00:30+00:00,1,False
7,2026-10-03 14:00:35+00:00,2,False
9,2026-10-03 14:00:45+00:00,1,False
10,2026-10-03 14:00:50+00:00,2,False
12,2026-10-03 14:01:00+00:00,1,False
13,2026-10-03 14:01:05+00:00,1,False


## Fake Query 3 — individual transaction attempts on `203.0.113.0/24`

Mirrors the BQ query that goes here:

```sql
SELECT txn_time, source_ip, CAST(amount AS FLOAT64) AS amount,
       accepted, rule_name, reason, account_name
FROM `voltdb-operator.beam_threat_detection.transactions`
WHERE source_ip LIKE '203.0.113.%'
  AND txn_time BETWEEN @start AND @end
ORDER BY txn_time
```

Hand-crafted dot distribution that lines up with the two per-window bar charts above:

- **5 ACCEPTED** dots in windows where BOTH `page_hits ≤ 500` AND `txn_count ≤ 20` (safe windows).
- **3 REJECTED by `SUBNET_PAGE_HIT_RATE`** in windows 7/9/10 (`page_hits > 500`, `txn_count < 20`) — pages caused the reject even though txn volume is normal.
- **4 REJECTED by `SUBNET_TXN_RATE`** all in window 15 (`txn_count = 24 > 20`, `page_hits = 6`) — the subnet-wide txn flood caused the reject, pages are quiet.
- **1 REJECTED by `VELOCITY_BURST`** in window 17, a safe window — showing that per-account rules fire independently of subnet rules (same account hit 6+ txns in 30s regardless of the surrounding subnet activity).

In [5]:
# Each row is one individual transaction attempt. Positions are chosen so that:
#   - ACCEPTED dots land only in windows with BOTH page_hits<=500 AND txn_count<=20
#   - SUBNET_PAGE_HIT_RATE rejects land in page-exceeded windows (w7, w9, w10)
#   - SUBNET_TXN_RATE rejects all land in w15 at times t+76..t+79 (above-threshold burst)
#
# Only the two SUBNET rules are shown on this chart — per-account rules
# (VELOCITY_BURST / HIGH_SPEND) belong on a separate per-account chart where
# their account-scoped context makes sense.

df_txns = pd.DataFrame([
    # --- 5 ACCEPTED (safe windows: w0, w3, w12, w19, w22) ---
    {"txn_time": T0 + timedelta(seconds=3),   "source_ip": "78.46.220.42",  "amount":   87.50, "accepted": 1, "rule_name": "NONE",                 "reason": "Accepted",                                                                     "account_name": "Alice Johnson"},
    {"txn_time": T0 + timedelta(seconds=17),  "source_ip": "78.46.220.61",  "amount":  249.99, "accepted": 1, "rule_name": "NONE",                 "reason": "Accepted",                                                                     "account_name": "Charlie Brown"},
    {"txn_time": T0 + timedelta(seconds=62),  "source_ip": "78.46.220.9",   "amount":   42.00, "accepted": 1, "rule_name": "NONE",                 "reason": "Accepted",                                                                     "account_name": "Eve Wilson"},
    {"txn_time": T0 + timedelta(seconds=97),  "source_ip": "78.46.220.112", "amount":  175.00, "accepted": 1, "rule_name": "NONE",                 "reason": "Accepted",                                                                     "account_name": "Alice Johnson"},
    {"txn_time": T0 + timedelta(seconds=112), "source_ip": "78.46.220.54",  "amount":  119.00, "accepted": 1, "rule_name": "NONE",                 "reason": "Accepted",                                                                     "account_name": "Bob Smith"},
    # --- 3 REJECTED by SUBNET_PAGE_HIT_RATE (page-exceeded windows: w7 @ t+37, w9 @ t+47, w10 @ t+52) ---
    {"txn_time": T0 + timedelta(seconds=37),  "source_ip": "78.46.220.199", "amount": 1250.00, "accepted": 0, "rule_name": "SUBNET_PAGE_HIT_RATE", "reason": "Too Many Page Hits from Subnet (>500/5s): count 850 from IP 78.46.220.199",    "account_name": "Bob Smith"},
    {"txn_time": T0 + timedelta(seconds=47),  "source_ip": "78.46.220.217", "amount":  980.00, "accepted": 0, "rule_name": "SUBNET_PAGE_HIT_RATE", "reason": "Too Many Page Hits from Subnet (>500/5s): count 1100 from IP 78.46.220.217",   "account_name": "Diana Prince"},
    {"txn_time": T0 + timedelta(seconds=52),  "source_ip": "78.46.220.88",  "amount":  420.00, "accepted": 0, "rule_name": "SUBNET_PAGE_HIT_RATE", "reason": "Too Many Page Hits from Subnet (>500/5s): count 980 from IP 78.46.220.88",     "account_name": "Charlie Brown"},
    # --- 4 REJECTED by SUBNET_TXN_RATE (all in w15, the txn-burst window) ---
    {"txn_time": T0 + timedelta(seconds=76),  "source_ip": "78.46.220.21",  "amount":  310.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE",      "reason": "Too Many Transactions from Subnet (>20/5s): count 21 from IP 78.46.220.21",    "account_name": "Alice Johnson"},
    {"txn_time": T0 + timedelta(seconds=77),  "source_ip": "78.46.220.34",  "amount":  125.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE",      "reason": "Too Many Transactions from Subnet (>20/5s): count 22 from IP 78.46.220.34",    "account_name": "Bob Smith"},
    {"txn_time": T0 + timedelta(seconds=78),  "source_ip": "78.46.220.78",  "amount":  440.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE",      "reason": "Too Many Transactions from Subnet (>20/5s): count 23 from IP 78.46.220.78",    "account_name": "Charlie Brown"},
    {"txn_time": T0 + timedelta(seconds=79),  "source_ip": "78.46.220.102", "amount":  215.00, "accepted": 0, "rule_name": "SUBNET_TXN_RATE",      "reason": "Too Many Transactions from Subnet (>20/5s): count 24 from IP 78.46.220.102",   "account_name": "Diana Prince"},
])
df_txns

# BQ VARIANT:
#
# subnet_prefix = ".".join(TARGET_SUBNET.split(".")[:3]) + "."
# df_txns = client.query(f"""
#     SELECT txn_time, source_ip, CAST(amount AS FLOAT64) AS amount,
#            accepted, rule_name, reason, account_name
#     FROM `voltdb-operator.beam_threat_detection.transactions`
#     WHERE source_ip LIKE '{subnet_prefix}%'
#       AND rule_name IN ('NONE', 'SUBNET_PAGE_HIT_RATE', 'SUBNET_TXN_RATE')
#       AND txn_time BETWEEN TIMESTAMP('{T0.isoformat()}')
#                        AND TIMESTAMP('{(T0 + timedelta(minutes=2)).isoformat()}')
#     ORDER BY txn_time
# """).to_dataframe()

,txn_time,source_ip,amount,accepted,rule_name,reason,account_name
0,2026-10-03 14:00:03+00:00,78.46.220.42,87.50,1,NONE,Accepted,Alice Johnson
1,2026-10-03 14:00:17+00:00,78.46.220.61,249.99,1,NONE,Accepted,Charlie Brown
2,2026-10-03 14:01:02+00:00,78.46.220.9,42.00,1,NONE,Accepted,Eve Wilson
3,2026-10-03 14:01:37+00:00,78.46.220.112,175.00,1,NONE,Accepted,Alice Johnson
4,2026-10-03 14:01:52+00:00,78.46.220.54,119.00,1,NONE,Accepted,Bob Smith
5,2026-10-03 14:00:37+00:00,78.46.220.199,1250.00,0,SUBNET_PAGE_HIT_RATE,Too Many Page Hits from Subnet (>500/5s): coun...,Bob Smith
6,2026-10-03 14:00:47+00:00,78.46.220.217,980.00,0,SUBNET_PAGE_HIT_RATE,Too Many Page Hits from Subnet (>500/5s): coun...,Diana Prince
7,2026-10-03 14:00:52+00:00,78.46.220.88,420.00,0,SUBNET_PAGE_HIT_RATE,Too Many Page Hits from Subnet (>500/5s): coun...,Charlie Brown
8,2026-10-03 14:01:16+00:00,78.46.220.21,310.00,0,SUBNET_TXN_RATE,Too Many Transactions from Subnet (>20/5s): co...,Alice Johnson
9,2026-10-03 14:01:17+00:00,78.46.220.34,125.00,0,SUBNET_TXN_RATE,Too Many Transactions from Subnet (>20/5s): co...,Bob Smith


## Fake Query 4 — per-subnet activity for the map

Aggregates page-hit and transaction-request counts per `/24` subnet across the whole chart time window, joined with GeoIP coordinates. Mirrors the BQ query that goes here (two subqueries, outer join, with GeoIP lookup from the enriched transactions table):

```sql
WITH page_counts AS (
  SELECT subnet, COUNT(*) AS page_count
  FROM `voltdb-operator.beam_threat_detection.page_hits`
  WHERE timestamp BETWEEN @start AND @end
  GROUP BY subnet
),
txn_counts AS (
  SELECT NET.IP_TO_STRING(NET.IP_TRUNC(NET.SAFE_IP_FROM_STRING(source_ip), 24)) AS subnet,
         COUNT(*)        AS txn_count,
         ANY_VALUE(country_name) AS country_name,
         ANY_VALUE(city_name)    AS city_name,
         ANY_VALUE(latitude)     AS lat,
         ANY_VALUE(longitude)    AS lon
  FROM `voltdb-operator.beam_threat_detection.transactions`
  WHERE txn_time BETWEEN @start AND @end
  GROUP BY subnet
)
SELECT COALESCE(p.subnet, t.subnet) AS subnet,
       t.country_name, t.city_name, t.lat, t.lon,
       COALESCE(p.page_count, 0) AS page_count,
       COALESCE(t.txn_count,  0) AS txn_count
FROM page_counts p
FULL OUTER JOIN txn_counts t USING (subnet)
```

Below: the attacker subnet (one giant blue marker, tiny orange one next to it) sits alongside a dozen baseline subnets around the world with balanced small blue + orange marker pairs.

In [6]:
# Per-subnet totals over the 2-minute window: 1 attacker + 11 worldwide baselines.
# The attacker shows high page activity (bot scan) AND elevated txn activity
# (the credential-stuffing burst), so it's "hot" on both axes. All baselines
# stay below the subnet-level hot thresholds.
#
# Subnet-level HOT thresholds (totals over the chart window — different from
# the per-5s rule thresholds that fire the actual rejections):
SUBNET_PAGE_HOT_COUNT = 500   # over the 2-min window — only attacker exceeds
SUBNET_TXN_HOT_COUNT  = 30    # over the 2-min window — only attacker exceeds

df_subnet_summary = pd.DataFrame([
    # --- Target / attacker subnet — dominant on both signals ---
    {"subnet": "78.46.220.0", "country": "Germany",      "city": "Nuremberg",   "lat": 49.4521, "lon": 11.0767, "page_count": 3617, "txn_count":  46},

    # --- Baseline subnets around the world — balanced small page+txn activity ---
    {"subnet": "78.46.1.0",   "country": "Germany",      "city": "Falkenstein", "lat": 50.4779, "lon": 12.3657, "page_count":   48, "txn_count":   7},
    {"subnet": "213.75.203.0","country": "Netherlands",  "city": "Amsterdam",   "lat": 52.3676, "lon":  4.9041, "page_count":   62, "txn_count":  11},
    {"subnet": "62.219.254.0","country": "Israel",       "city": "Haifa",       "lat": 32.7940, "lon": 34.9896, "page_count":   33, "txn_count":   5},
    {"subnet": "195.113.4.0", "country": "Czechia",      "city": "Prague",      "lat": 50.0755, "lon": 14.4378, "page_count":   85, "txn_count":  14},
    {"subnet": "202.12.29.0", "country": "Japan",        "city": "Tokyo",       "lat": 35.6762, "lon": 139.6503,"page_count":   27, "txn_count":   4},
    {"subnet": "156.154.70.0","country": "United States","city": "Sterling",    "lat": 39.0062, "lon": -77.4286,"page_count":  142, "txn_count":  18},
    {"subnet": "200.160.7.0", "country": "Brazil",       "city": "São Paulo",   "lat": -23.5505,"lon": -46.6333,"page_count":   56, "txn_count":   8},
    {"subnet": "143.244.32.0","country": "Singapore",    "city": "Singapore",   "lat":  1.3521, "lon": 103.8198,"page_count":   74, "txn_count":  12},
    {"subnet": "41.63.42.0",  "country": "South Africa", "city": "Johannesburg","lat": -26.2041,"lon": 28.0473, "page_count":   19, "txn_count":   3},
    {"subnet": "217.160.0.0", "country": "Germany",      "city": "Karlsruhe",   "lat": 49.0069, "lon":  8.4037, "page_count":   95, "txn_count":  15},
    {"subnet": "122.56.0.0",  "country": "New Zealand",  "city": "Auckland",    "lat": -36.8485,"lon": 174.7633,"page_count":   38, "txn_count":   6},
])
df_subnet_summary["is_page_hot"] = df_subnet_summary["page_count"] > SUBNET_PAGE_HOT_COUNT
df_subnet_summary["is_txn_hot"]  = df_subnet_summary["txn_count"]  > SUBNET_TXN_HOT_COUNT
# log-scale sizes make the attacker visible without flattening everything else.
df_subnet_summary["page_marker_size"] = df_subnet_summary["page_count"].apply(lambda n: max(math.log1p(n) * 6, 4))
df_subnet_summary["txn_marker_size"]  = df_subnet_summary["txn_count"].apply(lambda n: max(math.log1p(n) * 6, 4))
df_subnet_summary

,subnet,country,city,lat,lon,page_count,txn_count,is_page_hot,is_txn_hot,page_marker_size,txn_marker_size
0,78.46.220.0,Germany,Nuremberg,49.4521,11.0767,3617,46,True,True,49.162060,23.100886
1,78.46.1.0,Germany,Falkenstein,50.4779,12.3657,48,7,False,False,23.350922,12.476649
2,213.75.203.0,Netherlands,Amsterdam,52.3676,4.9041,62,11,False,False,24.858808,14.909440
3,62.219.254.0,Israel,Haifa,32.7940,34.9896,33,5,False,False,21.158163,10.750557
4,195.113.4.0,Czechia,Prague,50.0755,14.4378,85,14,False,False,26.726084,16.248301
5,202.12.29.0,Japan,Tokyo,35.6762,139.6503,27,4,False,False,19.993227,9.656627
6,156.154.70.0,United States,Sterling,39.0062,-77.4286,142,18,False,False,29.777068,17.666634
7,200.160.7.0,Brazil,São Paulo,-23.5505,-46.6333,56,8,False,False,24.258308,13.183347
8,143.244.32.0,Singapore,Singapore,1.3521,103.8198,74,12,False,False,25.904929,15.389696
9,41.63.42.0,South Africa,Johannesburg,-26.2041,28.0473,19,3,False,False,17.974394,8.317766


## The chart — geo view (top) + timeline (bottom)

In [7]:
# ── Color scheme (consistent map ↔ timeline) ────────────────────────────────
# Two color families, each with a light (normal) and dark (exceeded) shade:
#   Purple = page-hit side          Cyan (blue-leaning teal) = txn-request side
# No pure blue/orange — those read as "normal/not-normal" independently of legend.
COLOR_PAGE_LIGHT  = "#ce93d8"   # muted purple — normal page-hit line/marker
COLOR_PAGE_DARK   = "#6a1b9a"   # saturated purple — exceeded page line marker/threshold/reject-ring/shade
COLOR_TXN_LIGHT   = "#4dd0e1"   # muted cyan (more blue than teal) — normal txn-count line/marker
COLOR_TXN_DARK    = "#00838f"   # saturated cyan — exceeded txn marker/threshold/reject-ring/shade
COLOR_ACCEPT      = "#2e7d32"   # green — accepted txn dot (solid fill)
COLOR_REJECT_FILL = "#9e9e9e"   # grey — rejected txn dot interior (ring color encodes rule)
COLOR_HIGHLIGHT   = "#e91e63"   # magenta — attacker-subnet highlight ring on the geo map

RULE_RING_COLOR = {
    "SUBNET_PAGE_HIT_RATE": COLOR_PAGE_DARK,
    "SUBNET_TXN_RATE":      COLOR_TXN_DARK,
}

# Y positions on the right (txn-count) axis for individual txn dots.
DOT_Y_LOW = 3
DOT_Y_TXN_RATE = [21, 22, 23, 24]

ATTACKER = df_subnet_summary.iloc[0]  # first row — the attacker subnet
WINDOW_SECONDS = 5                     # one row per 5-second window

# The attacker's own page/txn counts dwarf baselines, so we DON'T plot its
# per-signal markers (they'd pile up with the magenta highlight ring and
# visually overwhelm the baseline pattern). The magenta ring + ⚠ ATTACKER
# label marks the attacker; the counts are still available via its hover.
df_baseline_subnets = df_subnet_summary.iloc[1:]   # skip row 0 = attacker

# Helper: contiguous (x_start, x_end) ranges where a boolean column is True.
# Used to turn per-window exceeded flags into shaded time-range rectangles.
def contiguous_true_ranges(df, flag_col, start_col, window_seconds=WINDOW_SECONDS):
    ranges = []
    start = None
    for i in range(len(df)):
        if df[flag_col].iloc[i] and start is None:
            start = df[start_col].iloc[i]
        elif not df[flag_col].iloc[i] and start is not None:
            end = df[start_col].iloc[i - 1] + timedelta(seconds=window_seconds)
            ranges.append((start, end))
            start = None
    if start is not None:
        end = df[start_col].iloc[-1] + timedelta(seconds=window_seconds)
        ranges.append((start, end))
    return ranges

# Helper: given per-window (x, y), extend to the step-function form that
# shape="hv" needs — a sentinel at (last_x + window_seconds, last_y) so the
# final window gets its horizontal segment.
def step_hv_series(df, x_col, y_col, window_seconds=WINDOW_SECONDS):
    xs = list(df[x_col]) + [df[x_col].iloc[-1] + timedelta(seconds=window_seconds)]
    ys = list(df[y_col]) + [df[y_col].iloc[-1]]
    return xs, ys

page_bad_periods = contiguous_true_ranges(df_hits,        "page_exceeded", "window_start")
txn_bad_periods  = contiguous_true_ranges(df_txn_windows, "txn_exceeded",  "window_start")

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.5, 0.5],
    vertical_spacing=0.14,
    specs=[
        [{"type": "geo"}],
        [{"type": "xy", "secondary_y": True}],
    ],
    subplot_titles=(
        f"<b>Geo view</b> — subnet activity worldwide (⚠ magenta ring marks the attacker on {ATTACKER['subnet']}/24)",
        f"<b>Zoom-in on {TARGET_SUBNET}/24 over 2 minutes</b> — when each subnet rule tripped and which transactions got rejected",
    ),
)

# ─────────────────────── Top panel: world map ────────────────────────────────
# 1. Attacker highlight ring (sole marker at the attacker's city)
fig.add_trace(go.Scattergeo(
    lat=[ATTACKER["lat"]], lon=[ATTACKER["lon"] - 2],
    marker=dict(
        size=52,
        color="rgba(0, 0, 0, 0)",            # fully transparent (ring only)
        line=dict(color=COLOR_HIGHLIGHT, width=2),
        symbol="circle",
    ),
    name="⚠ Attacker subnet",
    hovertemplate=(
        f"<b>⚠ Attacker</b><br>"
        f"subnet: {ATTACKER['subnet']}<br>"
        f"<span style='color:#6a1b9a'>■</span> page hits: {ATTACKER['page_count']:,}<br>"
        f"<span style='color:#00838f'>■</span> txn requests: {ATTACKER['txn_count']:,}"
        "<extra></extra>"
    ),
), row=1, col=1)

# 2a. Page markers — baselines only (attacker excluded; its activity dwarfs them
# visually and is already called out by the magenta ring + ⚠ label).
LON_OFFSET = 1.6
fig.add_trace(go.Scattergeo(
    lat=df_baseline_subnets["lat"],
    lon=df_baseline_subnets["lon"] - LON_OFFSET,
    marker=dict(
        size=df_baseline_subnets["page_marker_size"],
        color=COLOR_PAGE_LIGHT,
        opacity=0.85,
        line=dict(width=0.5, color="white"),
    ),
    name="Page hits per subnet (baseline)",
    customdata=df_baseline_subnets[["city", "country", "subnet", "page_count"]].values,
    hovertemplate=(
        "<b>%{customdata[0]}, %{customdata[1]}</b><br>"
        "subnet: %{customdata[2]}<br>"
        "<span style='color:#6a1b9a'>■</span> page hits: <b>%{customdata[3]:,}</b>"
        "<extra></extra>"
    ),
), row=1, col=1)

# 2b. Txn markers — baselines only, east-offset
fig.add_trace(go.Scattergeo(
    lat=df_baseline_subnets["lat"],
    lon=df_baseline_subnets["lon"] + LON_OFFSET,
    marker=dict(
        size=df_baseline_subnets["txn_marker_size"],
        color=COLOR_TXN_LIGHT,
        opacity=0.95,
        line=dict(width=0.5, color="white"),
    ),
    name="Txn requests per subnet (baseline)",
    customdata=df_baseline_subnets[["city", "country", "subnet", "txn_count"]].values,
    hovertemplate=(
        "<b>%{customdata[0]}, %{customdata[1]}</b><br>"
        "subnet: %{customdata[2]}<br>"
        "<span style='color:#00838f'>■</span> txn requests: <b>%{customdata[3]:,}</b>"
        "<extra></extra>"
    ),
), row=1, col=1)

# 3. "ATTACKER" text label — sits WELL ABOVE the magenta ring (no overlap)
fig.add_trace(go.Scattergeo(
    lat=[ATTACKER["lat"] + 20], lon=[ATTACKER["lon"] - 2],
    mode="text",
    text=["⚠ ATTACKER"],
    textfont=dict(color=COLOR_HIGHLIGHT, size=16, family="Segoe UI, sans-serif"),
    showlegend=False,
    hoverinfo="skip",
), row=1, col=1)

# ─────────────────────── Bottom panel: timeline ──────────────────────────────
# ── Shade bad time periods (drawn first so lines/dots sit on top) ───────────
for x0, x1 in page_bad_periods:
    fig.add_shape(
        type="rect",
        x0=x0, x1=x1, y0=0, y1=1, yref="y domain",
        fillcolor=COLOR_PAGE_DARK, opacity=0.12, line_width=0, layer="below",
        row=2, col=1,
    )
    fig.add_annotation(
        x=x0, xanchor="left",
        y=1, yanchor="top", yref="y domain",
        text="PAGE RATE exceeded", showarrow=False,
        font=dict(color=COLOR_PAGE_DARK, size=10),
        row=2, col=1,
    )
for x0, x1 in txn_bad_periods:
    fig.add_shape(
        type="rect",
        x0=x0, x1=x1, y0=0, y1=1, yref="y domain",
        fillcolor=COLOR_TXN_DARK, opacity=0.12, line_width=0, layer="below",
        row=2, col=1,
    )
    fig.add_annotation(
        x=x1, xanchor="right",
        y=1, yanchor="top", yref="y domain",
        text="TXN RATE exceeded", showarrow=False,
        font=dict(color=COLOR_TXN_DARK, size=10),
        row=2, col=1,
    )

# ── Page-hit STEP line (left axis, log scale) ───────────────────────────────
page_line_x, page_line_y = step_hv_series(df_hits, "window_start", "hits")
fig.add_trace(go.Scatter(
    x=page_line_x, y=page_line_y,
    mode="lines",
    line=dict(color=COLOR_PAGE_LIGHT, width=2.5, shape="hv"),
    name="Page hits per 5s",
    legendgroup="page",
    hoverinfo="skip",
), row=2, col=1, secondary_y=False)

page_marker_colors = [COLOR_PAGE_DARK if bad else COLOR_PAGE_LIGHT for bad in df_hits["page_exceeded"]]
fig.add_trace(go.Scatter(
    x=df_hits["window_start"], y=df_hits["hits"],
    mode="markers",
    marker=dict(size=8, color=page_marker_colors,
                line=dict(color=COLOR_PAGE_DARK, width=1)),
    name="Page hits per 5s",
    legendgroup="page",
    showlegend=False,
    hovertemplate="<b>%{y:,.0f} page hits</b> in 5s window starting %{x|%H:%M:%S}<extra></extra>",
), row=2, col=1, secondary_y=False)

# ── Txn-count STEP line (right axis, linear) ────────────────────────────────
txn_line_x, txn_line_y = step_hv_series(df_txn_windows, "window_start", "txn_count")
fig.add_trace(go.Scatter(
    x=txn_line_x, y=txn_line_y,
    mode="lines",
    line=dict(color=COLOR_TXN_LIGHT, width=2.5, shape="hv"),
    name="Txn requests per 5s",
    legendgroup="txn",
    hoverinfo="skip",
), row=2, col=1, secondary_y=True)

txn_marker_colors = [COLOR_TXN_DARK if bad else COLOR_TXN_LIGHT for bad in df_txn_windows["txn_exceeded"]]
fig.add_trace(go.Scatter(
    x=df_txn_windows["window_start"], y=df_txn_windows["txn_count"],
    mode="markers",
    marker=dict(size=8, color=txn_marker_colors,
                line=dict(color=COLOR_TXN_DARK, width=1)),
    name="Txn requests per 5s",
    legendgroup="txn",
    showlegend=False,
    hovertemplate="<b>%{y:,.0f} txn requests</b> in 5s window starting %{x|%H:%M:%S}<extra></extra>",
), row=2, col=1, secondary_y=True)

# ── Threshold lines + annotations (one per axis) ────────────────────────────
x_start = df_hits["window_start"].min()
x_end   = df_hits["window_start"].max() + timedelta(seconds=WINDOW_SECONDS)

fig.add_shape(
    type="line",
    x0=x_start, x1=x_end, y0=PAGE_RULE_THRESHOLD, y1=PAGE_RULE_THRESHOLD,
    line=dict(color=COLOR_PAGE_DARK, width=2, dash="dash"),
    row=2, col=1, secondary_y=False,
)
fig.add_annotation(
    x=x_start, y=PAGE_RULE_THRESHOLD,
    xanchor="left", yanchor="bottom",
    text=f"SUBNET_PAGE_HIT_RATE threshold ({PAGE_RULE_THRESHOLD} hits / 5s)",
    showarrow=False, font=dict(color=COLOR_PAGE_DARK, size=11),
    row=2, col=1, secondary_y=False,
)

fig.add_shape(
    type="line",
    x0=x_start, x1=x_end, y0=TXN_RULE_THRESHOLD, y1=TXN_RULE_THRESHOLD,
    line=dict(color=COLOR_TXN_DARK, width=2, dash="dash"),
    row=2, col=1, secondary_y=True,
)
fig.add_annotation(
    x=x_end, y=TXN_RULE_THRESHOLD,
    xanchor="right", yanchor="bottom",
    text=f"SUBNET_TXN_RATE threshold ({TXN_RULE_THRESHOLD} txns / 5s)",
    showarrow=False, font=dict(color=COLOR_TXN_DARK, size=11),
    row=2, col=1, secondary_y=True,
)

# ── Individual transaction dots ─────────────────────────────────────────────
df_txns_plot = df_txns.copy()
df_txns_plot["outcome"] = df_txns_plot["accepted"].map({1: "ACCEPTED", 0: "REJECTED"})

for (outcome, rule), grp in df_txns_plot.groupby(["outcome", "rule_name"]):
    if outcome == "ACCEPTED":
        ys = [DOT_Y_LOW] * len(grp)
        marker = dict(size=14, color=COLOR_ACCEPT, line=dict(color="white", width=2), symbol="circle")
        label = "Transaction — ACCEPTED"
    else:
        ys = DOT_Y_TXN_RATE[:len(grp)] if rule == "SUBNET_TXN_RATE" else [DOT_Y_LOW] * len(grp)
        ring = RULE_RING_COLOR.get(rule, "#424242")
        marker = dict(size=16, color=COLOR_REJECT_FILL, line=dict(color=ring, width=3), symbol="circle")
        label = f"Transaction — REJECTED ({rule})"
    fig.add_trace(go.Scatter(
        x=grp["txn_time"], y=ys,
        mode="markers",
        name=label,
        marker=marker,
        customdata=grp[["source_ip", "amount", "rule_name", "reason", "account_name"]].values,
        hovertemplate=(
            f"<b>%{{customdata[0]}}</b> — <b>{outcome}</b><br>"
            "amount=$%{customdata[1]:,.2f}<br>"
            "account=%{customdata[4]}<br>"
            "rule_name=%{customdata[2]}<br>"
            "reason=%{customdata[3]}"
            "<extra></extra>"
        ),
    ), row=2, col=1, secondary_y=True)

# ─────────────────────── Layout ──────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            "<b>Subnet-rate rules in action:</b> on "
            f"<span style='color:#e91e63'>{TARGET_SUBNET}/24</span>, "
            "<span style='color:#6a1b9a'>page-hit scan burst</span> and "
            "<span style='color:#00838f'>txn-request burst</span> each trip their own rule — "
            "<i>ProcessTransaction</i> rejects the matching txn attempts atomically.<br>"
            "<sup>Step line = per-5s counter value, dark marker = window exceeded threshold, shaded region = contiguous bad period; "
            "grey dots w/ colored ring = rejected (ring color encodes which rule fired).</sup>"
        ),
        x=0.02, xanchor="left", y=0.98,
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Segoe UI, sans-serif", size=12),
    legend=dict(
        orientation="h",
        yanchor="top", y=-0.08,
        xanchor="center", x=0.5,
    ),
    height=1000,
    margin=dict(l=70, r=70, t=140, b=170),
    geo=dict(
        projection_type="natural earth",
        showcoastlines=True,  coastlinecolor="#9aa0a6",
        showland=True,        landcolor="#e8f0fe",
        showcountries=True,   countrycolor="#dadce0",
        showocean=True,       oceancolor="#f1f3f4",
        bgcolor="white",
    ),
)
fig.layout.annotations[0].yshift = 12
fig.layout.annotations[1].yshift = 12
fig.update_xaxes(title_text="Time (UTC)", showgrid=True, gridcolor="#dadce0",
                 showline=True, linecolor="#9aa0a6", linewidth=1,
                 ticks="outside", tickcolor="#9aa0a6",
                 row=2, col=1)
fig.update_yaxes(
    title_text="Page hits per 5s (log)",
    type="log",
    range=[0, math.log10(2000)],
    showgrid=True, gridcolor="#dadce0",
    showline=True, linecolor="#9aa0a6", linewidth=1,
    ticks="outside", tickcolor="#9aa0a6",
    title_font=dict(color=COLOR_PAGE_DARK), tickfont=dict(color=COLOR_PAGE_DARK),
    row=2, col=1, secondary_y=False,
)
fig.update_yaxes(
    title_text="Transaction requests per 5s",
    type="linear",
    range=[0, 30],
    showgrid=False,
    showline=True, linecolor="#9aa0a6", linewidth=1,
    ticks="outside", tickcolor="#9aa0a6",
    tickmode="linear", tick0=0, dtick=5, tickformat="d",
    title_font=dict(color=COLOR_TXN_DARK), tickfont=dict(color=COLOR_TXN_DARK),
    row=2, col=1, secondary_y=True,
)
fig.show()

# ── Export the chart as PNG for blog embedding ────────────────────────────────
# Needs `kaleido` (installed in notebooks/.venv). The PNG goes to the
# synthetic/ folder so chart iterations don't blow away prior exports.
import os
_OUT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'notebooks', 'synthetic')
os.makedirs(_OUT_DIR, exist_ok=True)
_PNG = os.path.join(_OUT_DIR, 'chart1_subnet_rate_timeline.png')
fig.write_image(_PNG, width=1600, height=1000, scale=2)
print(f'wrote {_PNG}')


wrote /Users/mpopova/Marina/VoltProjects/voltdb-examples/beam-threat-detection/notebooks/synthetic/chart1_subnet_rate_timeline.png
